In [9]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve
from xgboost import XGBClassifier
from sklearn.base import clone

In [2]:
h = pd.read_parquet("../data/processed/arcep_2022_2025.parquet")

web = h.query("protocole == 'WEB'").copy()
web["echec"] = (1 - web["succes"]).astype(int)

NUMERIQUES = ["rsrp", "heure"]
BOOLEENNES = ["rsrp_absent"]
CATEGORIES = ["operator", "zone", "situation"]

X = web[NUMERIQUES + BOOLEENNES + CATEGORIES]
y = web["echec"]
annee = web["annee"]

entr = annee <= 2023      # entraînement
val  = annee == 2024      # validation (réglages)
test = annee == 2025      # test (une seule fois, à la fin)

for nom, masque in [("entraînement", entr), ("validation", val), ("test", test)]:
    print(f"{nom:13s} {masque.sum():>8,d} tests   taux d'échec {y[masque].mean():.2%}")

entraînement   750,585 tests   taux d'échec 10.03%
validation     248,289 tests   taux d'échec 8.33%
test           253,260 tests   taux d'échec 8.95%


In [3]:
def scores(y_vrai, proba, seuil=0.5):
    y_pred = (proba >= seuil).astype(int)
    return {
        "precision": round(precision_score(y_vrai, y_pred, zero_division=0), 3),
        "rappel":    round(recall_score(y_vrai, y_pred), 3),
        "f1":        round(f1_score(y_vrai, y_pred), 3),
        "ap":        round(average_precision_score(y_vrai, proba), 3),
    }

In [4]:
resultats = {}

# 1. Toujours prédire « succès » : probabilité d'échec nulle partout
resultats["majoritaire"] = scores(y[val], np.zeros(val.sum()))

# 2. Règle d'expert : échec si RSRP < -120 dBm ou absent
regle = ((X["rsrp"] < -120) | X["rsrp_absent"]).astype(int)
resultats["règle RSRP"] = scores(y[val], regle[val])

pd.DataFrame(resultats).T

,precision,rappel,f1,ap
majoritaire,0.000,0.000,0.000,0.083
règle RSRP,0.741,0.654,0.695,0.514


In [5]:
prep_lineaire = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("echelle", StandardScaler())]), NUMERIQUES),
    ("bool", "passthrough", BOOLEENNES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
])

logistique = Pipeline([
    ("prep", prep_lineaire),
    ("clf", LogisticRegression(max_iter=1000)),
])

logistique.fit(X[entr], y[entr])
proba_val = {"logistique": logistique.predict_proba(X[val])[:, 1]}
resultats["logistique"] = scores(y[val], proba_val["logistique"])

pd.DataFrame(resultats).T

,precision,rappel,f1,ap
majoritaire,0.000,0.000,0.000,0.083
règle RSRP,0.741,0.654,0.695,0.514
logistique,0.871,0.531,0.660,0.728


In [6]:
prep_arbres = ColumnTransformer([
    ("num", "passthrough", NUMERIQUES + BOOLEENNES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
])

modeles_arbres = {
    "forêt": RandomForestClassifier(n_estimators=100, min_samples_leaf=50,
                                    n_jobs=-1, random_state=0),
    "xgboost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                             n_jobs=-1, random_state=0),
}

for nom, clf in modeles_arbres.items():
    modele = Pipeline([("prep", prep_arbres), ("clf", clf)])
    modele.fit(X[entr], y[entr])
    modeles_arbres[nom] = modele
    proba_val[nom] = modele.predict_proba(X[val])[:, 1]
    resultats[nom] = scores(y[val], proba_val[nom])
    print(nom, "terminé")

pd.DataFrame(resultats).T

forêt terminé
xgboost terminé


,precision,rappel,f1,ap
majoritaire,0.000,0.000,0.000,0.083
règle RSRP,0.741,0.654,0.695,0.514
logistique,0.871,0.531,0.660,0.728
forêt,0.778,0.615,0.687,0.699
xgboost,0.785,0.611,0.687,0.709


In [8]:
seuils = {}
resultats_seuil = {"règle RSRP": resultats["règle RSRP"]}

for nom, p in proba_val.items():
    precision, rappel, s = precision_recall_curve(y[val], p)
    f1 = 2 * precision * rappel / (precision + rappel + 1e-12)
    meilleur = np.argmax(f1[:-1])
    seuils[nom] = s[meilleur]
    resultats_seuil[nom] = scores(y[val], p, seuils[nom]) | {"seuil": round(s[meilleur], 3)}

pd.DataFrame(resultats_seuil).T

,precision,rappel,f1,ap,seuil
règle RSRP,0.741,0.654,0.695,0.514,NaN
logistique,0.803,0.594,0.683,0.728,0.410
forêt,0.749,0.637,0.688,0.699,0.450
xgboost,0.761,0.632,0.690,0.709,0.456


In [10]:
modeles = {"logistique": logistique, **modeles_arbres}
entr_val = entr | val                                    # 2022-2024
moitie = np.random.default_rng(0).random(test.sum()) < 0.5
X_t, y_t = X[test], y[test]

final = {"règle RSRP": scores(y_t, regle[test])}
derive = {}
for nom, modele in modeles.items():
    m = clone(modele).fit(X[entr_val], y[entr_val])      # réentraîné sur 2022-2024
    p = m.predict_proba(X_t)[:, 1]
    final[nom] = scores(y_t, p, seuils[nom])

    m_2025 = clone(modele).fit(X_t[moitie], y_t[moitie])  # référence sans dérive
    p_2025 = m_2025.predict_proba(X_t[~moitie])[:, 1]
    derive[nom] = {"AP temporel": round(average_precision_score(y_t[~moitie], p[~moitie]), 3),
                   "AP intra-2025": round(average_precision_score(y_t[~moitie], p_2025), 3)}
    print(nom, "terminé")

display(pd.DataFrame(final).T)
pd.DataFrame(derive).T

logistique terminé
forêt terminé
xgboost terminé


,precision,rappel,f1,ap
règle RSRP,0.724,0.645,0.682,0.499
logistique,0.793,0.564,0.659,0.720
forêt,0.767,0.609,0.679,0.701
xgboost,0.775,0.593,0.672,0.700


,AP temporel,AP intra-2025
logistique,0.718,0.725
forêt,0.697,0.749
xgboost,0.696,0.753


In [11]:
pd.DataFrame(final).T.to_csv("../results/tables/web_classification_test_2025.csv")
pd.DataFrame(derive).T.to_csv("../results/tables/web_classification_derive.csv")
pd.Series(seuils).round(3).to_csv("../results/tables/web_classification_seuils.csv", header=["seuil"])